# Music Sales from 1973 to 2013: an EDA

### First step: looking at the data

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
music_raw = pd.read_csv('musicdata/musicdata.csv')
music = music_raw.copy()
music.info()

<class 'pandas.DataFrame'>
RangeIndex: 3008 entries, 0 to 3007
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   format             3008 non-null   str    
 1   metric             3008 non-null   str    
 2   year               3008 non-null   int64  
 3   number_of_records  3008 non-null   int64  
 4   value_actual       1351 non-null   float64
dtypes: float64(1), int64(2), str(2)
memory usage: 117.6 KB


In [4]:
music.sample(10)

,format,metric,year,number_of_records,value_actual
2976,Vinyl Single,Value (Adjusted),1988,1,389.860717
1693,Ringtones & Ringbacks,Value,1974,1,NaN
857,CD,Value,1984,1,103.300000
2309,Download Single,Value (Adjusted),1979,1,NaN
1448,Music Video (Physical),Value,2011,1,151.000000
2686,Other Tapes,Value (Adjusted),1980,1,NaN
1724,Ringtones & Ringbacks,Value,2005,1,421.600000
623,Kiosk,Units,1985,1,NaN
2394,Kiosk,Value (Adjusted),2017,1,2.351668
1327,LP/EP,Value,1984,1,1548.800000


In [5]:
music['format'].unique()

<StringArray>
[                                'CD',                          'CD Single',
                           'Cassette',                    'Cassette Single',
                              'LP/EP',                       'Vinyl Single',
                          '8 - Track',                        'Other Tapes',
             'Music Video (Physical)',                          'DVD Audio',
                               'SACD',                    'Download Single',
                     'Download Album',                              'Kiosk',
               'Download Music Video',              'Ringtones & Ringbacks',
                 'Paid Subscriptions',     'Limited Tier Paid Subscription',
 'On-Demand Streaming (Ad-Supported)',       'Other Ad-Supported Streaming',
                      'Other Digital',                  'Paid Subscription',
        'SoundExchange Distributions',                    'Synchronization']
Length: 24, dtype: str

In [6]:
music['number_of_records'].unique()

array([1])

### Second step: cleaning the data

We don't need the number_of_records column as it is 1 everywhere. We shall drop it.

In [7]:
music.drop(columns='number_of_records',inplace=True)
music.info()

<class 'pandas.DataFrame'>
RangeIndex: 3008 entries, 0 to 3007
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   format        3008 non-null   str    
 1   metric        3008 non-null   str    
 2   year          3008 non-null   int64  
 3   value_actual  1351 non-null   float64
dtypes: float64(1), int64(1), str(2)
memory usage: 94.1 KB


Now we check for NaNs.

In [22]:
music.info()

<class 'pandas.DataFrame'>
Index: 1351 entries, 10 to 3007
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   format             1351 non-null   str    
 1   metric             1351 non-null   str    
 2   year               1351 non-null   int64  
 3   number_of_records  1351 non-null   int64  
 4   value_actual       1351 non-null   float64
dtypes: float64(1), int64(2), str(2)
memory usage: 63.3 KB


In [12]:
music[music['value_actual'].isna()].head(20)

,format,metric,year,number_of_records,value_actual
0,CD,Units,1973,1,NaN
1,CD,Units,1974,1,NaN
2,CD,Units,1975,1,NaN
3,CD,Units,1976,1,NaN
4,CD,Units,1977,1,NaN
5,CD,Units,1978,1,NaN
6,CD,Units,1979,1,NaN
7,CD,Units,1980,1,NaN
8,CD,Units,1981,1,NaN
9,CD,Units,1982,1,NaN


In [15]:
music[music['year']==1973]

,format,metric,year,number_of_records,value_actual
0,CD,Units,1973,1,NaN
47,CD Single,Units,1973,1,NaN
94,Cassette,Units,1973,1,15.000000
141,Cassette Single,Units,1973,1,NaN
188,LP/EP,Units,1973,1,280.000000
...,...,...,...,...,...
2773,Ringtones & Ringbacks,Value (Adjusted),1973,1,NaN
2820,SACD,Value (Adjusted),1973,1,NaN
2867,SoundExchange Distributions,Value (Adjusted),1973,1,NaN
2914,Synchronization,Value (Adjusted),1973,1,NaN


The only NaN values are in value_actual, that is the column containing the actual value of the metric. Therefore NaN values here have no use in our analysis and cannot be replaced with a relevant estimate since they represent missing data. We shall drop them.

In [16]:
music = music[~music['value_actual'].isna()]
music.info()

<class 'pandas.DataFrame'>
Index: 1351 entries, 10 to 3007
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   format             1351 non-null   str    
 1   metric             1351 non-null   str    
 2   year               1351 non-null   int64  
 3   number_of_records  1351 non-null   int64  
 4   value_actual       1351 non-null   float64
dtypes: float64(1), int64(2), str(2)
memory usage: 63.3 KB


In [20]:
music[(music['year']==1973) & (music['format']=='Cassette')]

,format,metric,year,number_of_records,value_actual
94,Cassette,Units,1973,1,15.000000
940,Cassette,Value,1973,1,76.000000
2068,Cassette,Value (Adjusted),1973,1,437.611081


In [21]:
music[(music['year']==1973) & (music['format']=='LP/EP')]

,format,metric,year,number_of_records,value_actual
188,LP/EP,Units,1973,1,280.000000
1316,LP/EP,Value,1973,1,1246.000000
2397,LP/EP,Value (Adjusted),1973,1,7174.518514


For each year, every format type has 3 different metrics: units, value and value (adjusted). Since we want to compare economic data coming from different periods of time, we can either deal with absolute units (i.e. volumes of sells) or the value adjusted for inflation. The latter is the best, for looking at units alone would not take into account any cost variation that is unrelated to inflation.